# Phase 0: Transformer from Scratch
## Entrenando un transformer en Shakespeare

En este notebook implementaremos un transformer minimalista desde cero y lo entrenaremos en el dataset de Shakespeare.

**Objetivos:**
1. Entender cómo funcionan los transformers a nivel de código
2. Implementar embedding, positional encoding, y self-attention
3. Entrenar el modelo en language modeling
4. Generar texto de forma autónoma

## Celda 1: Imports y Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path

# Detectar device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

### Notas sobre setup:
- MPS es Metal Performance Shaders (GPU de M4 Pro)
- Si MPS no está disponible, fallback a CPU

**Tus anotaciones aquí:**

## Celda 2: Cargar datos

In [3]:
# Cargar el dataset completo
with open('../data/raw/shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Dataset size: {len(text):,} characters")
print(f"Unique characters: {len(set(text))}")
print(f"\nFirst 300 chars:\n{text[:300]}")

Dataset size: 1,115,394 characters
Unique characters: 65

First 300 chars:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


### Análisis del dataset:
- Tamaño total: ~1 MB
- Caracteres únicos: ~65 (a-z, A-Z, números, puntuación, espacios)
- Esto es "character-level" tokenización: cada token es un carácter

**Tus anotaciones aquí:**

## Celda 3: Tokenización (character-level)

In [4]:
# Crear vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(f"Vocabulary size: {vocab_size}")
print(f"Characters in vocab: {chars}")

# Crear mappings
stoi = {ch: i for i, ch in enumerate(chars)}  # string to int
itos = {i: ch for i, ch in enumerate(chars)}  # int to string

# Funciones helper
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# Test
test_string = "Hello, World!"
encoded = encode(test_string)
decoded = decode(encoded)

print(f"\nTest encode/decode:")
print(f"  Original: '{test_string}'")
print(f"  Encoded: {encoded}")
print(f"  Decoded: '{decoded}'")
print(f"  Match: {test_string == decoded}")

Vocabulary size: 65
Characters in vocab: ['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

Test encode/decode:
  Original: 'Hello, World!'
  Encoded: [20, 43, 50, 50, 53, 6, 1, 35, 53, 56, 50, 42, 2]
  Decoded: 'Hello, World!'
  Match: True


### Explicación de tokenización:
- **stoi** (string to int): mapea cada carácter a un número (0-64)
- **itos** (int to string): mapea números de vuelta a caracteres
- **encode**: convierte texto a números
- **decode**: convierte números de vuelta a texto

Este es el nivel más básico de tokenización. Los LLMs reales usan BPE (Byte Pair Encoding) o SentencePiece.

**Tus anotaciones aquí:**

## Celda 4: Dataset

In [5]:
class ShakespeareDataset(Dataset):
    """Dataset para Shakespeare con tokenización character-level"""
    def __init__(self, text, stoi, block_size=256):
        self.data = torch.tensor(encode(text), dtype=torch.long)
        self.block_size = block_size
    
    def __len__(self):
        # Cuántas secuencias podemos crear
        return len(self.data) - self.block_size
    
    def __getitem__(self, idx):
        # Retorna (X, Y) donde Y es el siguiente token en cada posición
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]
        return x, y

# Crear splits: 90% train, 10% val
split_idx = int(0.9 * len(text))
train_text = text[:split_idx]
val_text = text[split_idx:]

print(f"Train/val split:")
print(f"  Train: {len(train_text):,} chars ({100*len(train_text)/len(text):.1f}%)")
print(f"  Val: {len(val_text):,} chars ({100*len(val_text)/len(text):.1f}%)")

# Hiperparámetros
block_size = 256  # Longitud de la secuencia
batch_size = 32   # Cuántas secuencias por batch

# Crear datasets
train_dataset = ShakespeareDataset(train_text, stoi, block_size=block_size)
val_dataset = ShakespeareDataset(val_text, stoi, block_size=block_size)

# Crear dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=0)

print(f"\nDataLoaders:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Each batch: {batch_size} sequences of length {block_size}")

# Inspeccionar un batch
x, y = next(iter(train_loader))
print(f"\nSample batch shape:")
print(f"  X: {x.shape}  (batch_size, seq_length)")
print(f"  Y: {y.shape}  (batch_size, seq_length)")
print(f"\nFirst sequence, first 20 chars:")
print(f"  X: {x[0, :20].tolist()} → '{decode(x[0, :20].tolist())}'")
print(f"  Y: {y[0, :20].tolist()} → '{decode(y[0, :20].tolist())}'")

Train/val split:
  Train: 1,003,854 chars (90.0%)
  Val: 111,540 chars (10.0%)

DataLoaders:
  Train batches: 31363
  Val batches: 3478
  Each batch: 32 sequences of length 256

Sample batch shape:
  X: torch.Size([32, 256])  (batch_size, seq_length)
  Y: torch.Size([32, 256])  (batch_size, seq_length)

First sequence, first 20 chars:
  X: [56, 53, 51, 1, 58, 46, 43, 52, 41, 43, 6, 0, 31, 39, 63, 47, 52, 45, 6, 1] → 'rom thence,
Saying, '
  Y: [53, 51, 1, 58, 46, 43, 52, 41, 43, 6, 0, 31, 39, 63, 47, 52, 45, 6, 1, 46] → 'om thence,
Saying, h'


### Explicación de Dataset:
- **block_size**: cuántos tokens ve el modelo a la vez (contexto)
- **X**: secuencia de entrada (tokens 0-255)
- **Y**: targets (tokens 1-256) - el siguiente token para cada posición

Esto es **autoregressive language modeling**: predecir el siguiente token dado los anteriores.

**Tus anotaciones aquí:**

## Celda 5: Positional Encoding

In [6]:
class PositionalEncoding(nn.Module):
    """Positional encoding sinusoidal (del paper original de Transformers)"""
    def __init__(self, d_model, max_len=2048):
        super().__init__()
        
        # Crear matriz de posiciones
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # Término para la escala de frecuencias
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            -(np.log(10000.0) / d_model))
        
        # Aplicar sin y cos alternados
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model % 2 == 1:
            pe[:, 1::2] = torch.cos(position * div_term[:-1])
        else:
            pe[:, 1::2] = torch.cos(position * div_term)
        
        # Registrar como buffer (no es parámetro, solo constante)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1), :]

# Test
pe = PositionalEncoding(d_model=256, max_len=512)
print(f"Positional encoding shape: {pe.pe.shape}")
print(f"PE values (posición 0, primeros 10 dims):")
print(f"  {pe.pe[0, 0, :10]}")
print(f"\nPE values (posición 1, primeros 10 dims):")
print(f"  {pe.pe[0, 1, :10]}")

Positional encoding shape: torch.Size([1, 512, 256])
PE values (posición 0, primeros 10 dims):
  tensor([0., 1., 0., 1., 0., 1., 0., 1., 0., 1.])

PE values (posición 1, primeros 10 dims):
  tensor([0.8415, 0.5403, 0.8020, 0.5974, 0.7617, 0.6479, 0.7214, 0.6925, 0.6816,
        0.7318])


### Explicación de Positional Encoding:
- Los transformers no ven el orden (self-attention es permutation-invariant)
- Positional encoding añade información de posición
- Usa seno y coseno en diferentes frecuencias
- Se suma al embedding de entrada

**Tus anotaciones aquí:**

## Celda 6: Transformer Block

In [7]:
class TransformerBlock(nn.Module):
    """Un bloque transformer: self-attention + feed-forward"""
    def __init__(self, d_model, n_heads, dim_feedforward=1024, dropout=0.1):
        super().__init__()
        
        # Multi-head self-attention
        self.self_attn = nn.MultiheadAttention(
            d_model, n_heads, dropout=dropout, batch_first=True
        )
        
        # Feed-forward network (MLP)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x shape: (batch, seq_len, d_model)
        
        # Self-attention con residual connection
        attn_out, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward con residual connection
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        
        return x

# Test
block = TransformerBlock(d_model=256, n_heads=4, dim_feedforward=1024)
x_test = torch.randn(2, 10, 256)  # (batch=2, seq=10, d_model=256)
out = block(x_test)
print(f"Input shape: {x_test.shape}")
print(f"Output shape: {out.shape}")
print(f"✓ Transformer block works!")

Input shape: torch.Size([2, 10, 256])
Output shape: torch.Size([2, 10, 256])
✓ Transformer block works!


### Explicación de Transformer Block:
**Componentes:**
1. **Self-attention**: cada token atiende a todos los otros tokens
2. **Residual connections**: x + attention(x) ayuda con gradientes
3. **Layer norm**: normaliza activaciones (estabiliza training)
4. **Feed-forward**: MLP aplicado a cada token independientemente

Esta estructura es el corazón de los transformers modernos.

**Tus anotaciones aquí:**

## Celda 7: Modelo Completo

In [8]:
class SimpleTransformer(nn.Module):
    """Transformer completo para language modeling"""
    def __init__(self, vocab_size, d_model=256, n_heads=4, n_layers=2, 
                 block_size=256, dropout=0.1):
        super().__init__()
        
        # Embedding
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len=block_size)
        
        # Stack de transformer blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dim_feedforward=4*d_model, dropout=dropout)
            for _ in range(n_layers)
        ])
        
        # Output layer
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        # x shape: (batch, seq_len)
        
        # Token embedding + positional encoding
        x = self.token_embedding(x)
        x = self.pos_encoding(x)
        
        # Pasar por transformer blocks
        for block in self.transformer_blocks:
            x = block(x)
        
        # Final output
        x = self.final_norm(x)
        logits = self.lm_head(x)  # (batch, seq_len, vocab_size)
        
        return logits

# Crear modelo
model = SimpleTransformer(
    vocab_size=vocab_size,
    d_model=256,
    n_heads=4,
    n_layers=2,
    block_size=block_size,
    dropout=0.1
).to(device)

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model architecture:")
print(f"  Vocab size: {vocab_size}")
print(f"  D_model: 256")
print(f"  N_heads: 4")
print(f"  N_layers: 2")
print(f"  Block size: {block_size}")
print(f"\nModel parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"\nModel summary:")
print(model)

Model architecture:
  Vocab size: 65
  D_model: 256
  N_heads: 4
  N_layers: 2
  Block size: 256

Model parameters:
  Total: 1,613,377
  Trainable: 1,613,377

Model summary:
SimpleTransformer(
  (token_embedding): Embedding(65, 256)
  (pos_encoding): PositionalEncoding()
  (transformer_blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (ffn): Sequential(
        (0): Linear(in_features=256, out_features=1024, bias=True)
        (1): GELU(approximate='none')
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=1024, out_features=256, bias=True)
      )
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (final_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
 

### Arquitectura del modelo:
- **Embedding layer**: convierte tokens a vectores de 256 dimensiones
- **Positional encoding**: suma información de posición
- **2 transformer blocks**: cada uno con self-attention (4 heads) + FFN
- **LM head**: proyecto de 256 dims de vuelta a vocab_size (65)

Total: ~500k parámetros (pequeño, pero funcional)

**Tus anotaciones aquí:**

## Celda 8: Test Forward Pass

In [9]:
# Test forward pass
x_test, y_test = next(iter(train_loader))
x_test = x_test.to(device)

logits = model(x_test)

print(f"Forward pass test:")
print(f"  Input shape: {x_test.shape}")
print(f"  Output shape: {logits.shape}")
print(f"  Output range: [{logits.min():.2f}, {logits.max():.2f}]")
print(f"\n✓ Forward pass successful!")

# Calcular loss de ejemplo
criterion = nn.CrossEntropyLoss()
y_test = y_test.to(device)
loss = criterion(logits.view(-1, vocab_size), y_test.view(-1))
print(f"\nSample loss: {loss.item():.4f}")
print(f"(Esperado ~{np.log(vocab_size):.2f} al inicio, random)")

Forward pass test:
  Input shape: torch.Size([32, 256])
  Output shape: torch.Size([32, 256, 65])
  Output range: [-2.36, 2.42]

✓ Forward pass successful!

Sample loss: 4.2639
(Esperado ~4.17 al inicio, random)


**Tus anotaciones aquí:**

## Celda 9: Training Loop

In [10]:
# Crear optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

def train_epoch(epoch_num):
    """Entrenar una época"""
    model.train()
    total_loss = 0
    num_batches = 0
    
    for batch_idx, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        
        # Forward pass
        logits = model(x)  # shape: (batch_size, block_size, vocab_size)
        
        # Reshape para cross_entropy
        loss = criterion(
            logits.view(-1, vocab_size),  # (batch_size * block_size, vocab_size)
            y.view(-1)                      # (batch_size * block_size,)
        )
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Gradient clipping
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        
        if batch_idx % 50 == 0 and batch_idx > 0:
            avg_loss = total_loss / num_batches
            print(f"  Batch {batch_idx:3d}/{len(train_loader)}: loss={loss.item():.4f} (avg={avg_loss:.4f})")
    
    avg_loss = total_loss / num_batches
    return avg_loss

### Explicación de training:
- **Forward**: pasar batch por el modelo
- **Loss**: cross_entropy entre logits y targets
- **Backward**: calcular gradientes
- **Gradient clipping**: evitar explosiones de gradientes (importante en RNNs/transformers)
- **Step**: actualizar pesos con optimizer

**Tus anotaciones aquí:**

## Celda 10: Validation Loop

In [11]:
@torch.no_grad()
def evaluate(data_loader):
    """Evaluar en un dataset completo"""
    model.eval()
    total_loss = 0
    num_batches = 0
    
    for x, y in data_loader:
        x, y = x.to(device), y.to(device)
        
        logits = model(x)
        loss = criterion(logits.view(-1, vocab_size), y.view(-1))
        
        total_loss += loss.item()
        num_batches += 1
    
    avg_loss = total_loss / num_batches
    return avg_loss

**Tus anotaciones aquí:**

## Celda 11: ENTRENAR (EJECUTA ESTO)

In [13]:
print("\n" + "="*60)
print("ENTRENAMIENTO")
print("="*60)

num_epochs = 5
best_val_loss = float('inf')

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 60)
    
    # Train
    train_loss = train_epoch(epoch)
    
    # Validate
    val_loss = evaluate(val_loader)
    
    print(f"\n  Train loss: {train_loss:.4f}")
    print(f"  Val loss:   {val_loss:.4f}")
    
    # Guardar checkpoint si mejoró
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        checkpoint_dir = Path('results/checkpoints/phase0_shakespeare')
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': val_loss,
        }, checkpoint_dir / 'best_model.pt')
        print(f"  ✓ Best model saved! (val_loss={val_loss:.4f})")

print("\n" + "="*60)
print(f"Entrenamiento completado!")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*60)


ENTRENAMIENTO

Epoch 1/5
------------------------------------------------------------


KeyboardInterrupt: 

### Resultado esperado:
- Época 1: train_loss ~4.0, val_loss ~3.8
- Época 5: train_loss ~1.5, val_loss ~2.0

El modelo aprende a predecir Shakespeare!

**Tus anotaciones aquí:**

## Celda 12: Generación de Texto

In [ ]:
@torch.no_grad()
def generate(prompt, max_length=200, temperature=0.8, top_k=None):
    """Generar texto dado un prompt"""
    model.eval()
    
    # Encode prompt
    x = torch.tensor([encode(prompt)], dtype=torch.long).to(device)
    
    generated = list(prompt)
    
    for _ in range(max_length):
        # Usar últimos block_size tokens
        x_input = x[:, -block_size:]
        
        # Forward
        logits = model(x_input)
        logits = logits[:, -1, :] / temperature  # Aplicar temperatura
        
        # Top-k sampling (opcional)
        if top_k is not None:
            indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
            logits[indices_to_remove] = float('-inf')
        
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        x = torch.cat([x, next_token], dim=-1)
        generated.append(itos[next_token.item()])
    
    return ''.join(generated)

# Generar ejemplos
print("\n" + "="*60)
print("GENERACIÓN DE TEXTO")
print("="*60)

prompts = [
    "To be or not",
    "All the world's",
    "The quick brown"
]

for prompt in prompts:
    print(f"\nPrompt: '{prompt}'")
    print("-" * 60)
    generated = generate(prompt, max_length=150, temperature=0.8)
    print(generated)
    print()

### Explicación de generación:
- **Temperature**: controla aleatoriedad (< 1 = más determinista, > 1 = más aleatorio)
- **Top-k sampling**: solo muestrear entre los k tokens más probables
- **Autoregressive**: generar un token a la vez, condicionado en el historial

**Tus anotaciones aquí:**

## Celda 13: Análisis Final

In [ ]:
print("\n" + "="*60)
print("RESUMEN - PHASE 0 COMPLETADO")
print("="*60)

print(f"""
✓ Implementamos un transformer desde cero
✓ Entrenamos en Shakespeare
✓ Generamos texto de forma autónoma

Lo que aprendiste:
  1. Character-level tokenization
  2. Token embeddings + positional encoding
  3. Self-attention mechanism
  4. Transformer blocks con residual connections
  5. Autoregressive language modeling
  6. Inference con sampling

Próximos pasos (Phase 1):
  - Fine-tune Llama 2 7B (modelo real de HuggingFace)
  - Usar LoRA para ajuste eficiente
  - Evaluar en tareas reales (HumanEval, perplexity)
  - Benchmark vs baselines

Para ejecutar Phase 1:
  python src/experiments/baseline_llama.py
""")

## Notas Finales

**Diferencias con LLMs reales:**
- Nuestro modelo: 500k params, character-level
- Llama 2 7B: 7 billion params, BPE tokenization
- GPT-4: 175+ billion params, advanced tokenization

Pero la arquitectura base es la misma. Transformers escalan bien.

**Recursos para profundizar:**
- "Attention is All You Need" paper (2017)
- Karpathy's nanoGPT
- HuggingFace documentation
- Nested Learning paper (Google, 2025)

**Tus anotaciones finales:**